# Minimal AIDA Usage

This notebook shows the smallest public usage path for AIDA.

In [ ]:
# %pip install -e ..

In [7]:
from pathlib import Path
import sys
import pandas as pd

repo_root = Path.cwd().resolve().parent.parent
public_src = repo_root / "public" / "src"
if str(public_src) not in sys.path:
    sys.path.insert(0, str(public_src))

from aida import AIDA

In [8]:
csv_path = Path("airline-safety.csv")
if not csv_path.exists():
    df = pd.read_csv("https://raw.githubusercontent.com/fivethirtyeight/data/master/airline-safety/airline-safety.csv")
    df.to_csv(csv_path, index=False)
goal = "Find the most decision-useful patterns in airline safety incidents."

In [9]:
context = AIDA.build_context(csv_path=csv_path, goal=goal, flag_id="airline-safety")
context.row_count, context.schema[:5]

(56,
 [{'column': 'airline', 'dtype': 'str'},
  {'column': 'avail_seat_km_per_week', 'dtype': 'int64'},
  {'column': 'incidents_85_99', 'dtype': 'int64'},
  {'column': 'fatal_accidents_85_99', 'dtype': 'int64'},
  {'column': 'fatalities_85_99', 'dtype': 'int64'}])

In [10]:
# Set GEMINI_API_KEY before running.
result,reviewed_insights = AIDA.run(
    csv_path=csv_path,
    goal=goal,
    model_name="gemini/gemini-flash-lite-latest",
    rounds=2,
    questions_per_round=3,
    with_review=False,
)

In [11]:
result["final_relevant_insights"][:5]

[{'insight': 'Airline scale is a primary predictor of incident frequency but is a poor predictor of fatality risk.',
  'evidence': 'Airlines with higher available seat kilometers per week are strongly correlated with higher incident counts (correlation of 0.73), while the correlation between scale and fatalities is significantly lower (0.23).',
  'question_answered': "Is there a strong correlation between an airline's scale and its safety outcomes (incidents vs. fatalities)?"},
 {'evidence': 'The 5 airlines with the largest reduction in incidents (85-99 to 00-14) reduced incidents by an average of 23.8 per carrier; however, fatalities do not follow a uniform downward trend, as seen with Air France, which reduced incidents by 8 but saw a fatality increase of 258.',
  'question_answered': 'Do airlines with the largest reductions in incidents also show the largest reductions in fatalities?',
  'insight': 'Reductions in incident frequency do not consistently translate to reductions in fata